# L6c: Modern Hopfield Networks and the Single-Head Attention Mechanism
In the previous lecture, we introduced classical Hopfield networks as associative memory systems that encode binary patterns using Hebbian learning. While elegant, classical networks have significant limitations in storage capacity and pattern representation.

In this lecture, we extend our understanding to modern Hopfield networks, which overcome these limitations and provide a bridge to contemporary deep learning architectures.

> __Learning Objectives:__
> 
> By the end of this lecture, you should be able to define and explain:
>
> * __Modern Hopfield Energy Function__: Explain how the log-sum-exp energy function generalizes the classical Hopfield energy to enable storage of continuous (not just binary) patterns. Understand the role of the inverse temperature parameter $\beta$ in controlling convergence sharpness and the connection between log-sum-exp and softmax operations.
> * __Exponential Storage and Convergence__: Describe how modern Hopfield networks achieve exponentially larger storage capacity compared to the classical $0.138N$ limit. Explain the memory retrieval algorithm using softmax-weighted updates and understand why modern networks exhibit exponential convergence (typically 1–5 iterations) rather than the polynomial convergence of classical networks.
> * __Connection to Attention Mechanisms__: Recognize that the modern Hopfield update rule is mathematically equivalent to single-head attention in transformer architectures. Understand how queries, keys, and values in attention correspond to states and memories in Hopfield networks, bridging associative memory theory with modern deep learning.

Let's get started!
___

## Examples
Today, we will use the following examples to illustrate key concepts:

> [▶ Analyze a modern Hopfield network](CHEME-5800-L15c-Example-ModernHopfieldNetwork-Fall-2025.ipynb). In this example, we analyze a modern Hopfield Network to understand how it encodes and retrieves gray-scale (continuous) patterns. This example builds on the concepts introduced in the previous lecture and demonstrates the application of modern Hopfield networks in continuous associative memory tasks.
___

## Idea: Modern Hopfield Networks
Classical Hopfield networks are elegant but limited: they can only store a small number of binary patterns reliably. Modern Hopfield networks address these shortcomings.

> __Why are these interesting?__ 
> 
> The modern Hopfield energy function generalizes that of the classical network to allow the storage of *continuous* (not just binary) patterns. It also enables the storage of exponentially many (potentially *correlated*) memories and exhibits much faster convergence behavior than classical networks.

The key innovation in modern Hopfield networks is the reformulation of the energy function. [Krotov and Hopfield (2016)](https://arxiv.org/abs/1606.01164) proposed a new energy function of the form:
$$
\begin{align*}
E(\mathbf{s}) &= -\sum_{i=1}^{K}F(\underbrace{\mathbf{m}_{i}^{\top}\mathbf{s}}_{\text{similarity}}) \\
& = -\sum_{i=1}^{K}F(\langle\mathbf{m}_{i},\mathbf{s}\rangle) \\
\end{align*}
$$
where $F:\mathbb{R}\to\mathbb{R}$ is a nonlinear function, $\mathbf{m}_i$ is the $i$-th memory, $K$ is the number of memories, and $\mathbf{s}$ is the state of the network. 
The function $F$ maps the similarity (inner product) between the state and memory vectors to a scalar energy value. The choice of $F$ determines the type of memory dynamics and convergence behavior. There are many choices for $F$, but one particularly interesting choice was proposed by [Ramsauer et al. (2020)](https://arxiv.org/abs/2008.02217):
$$
\begin{align*}
E(\mathbf{s}) &= -\texttt{lse}(\beta,\mathbf{X}^{\top}\mathbf{s}) + \frac{1}{2}\mathbf{s}^{\top}\mathbf{s} + \frac{1}{\beta}\log(K)+ \frac{1}{2}M^{2} \\
\end{align*}
$$
where $\mathbf{X}\in\mathbb{R}^{N\times{K}}$ is the matrix of memories, i.e., each memory $\mathbf{m}_{1},\dots,\mathbf{m}_{K}$ consisting of $N$ features is a column of the matrix, $\mathbf{s}$ is the current state of the network, and $\texttt{lse}(\cdot)$ is the log-sum-exp function:
$$
\begin{align*}
\texttt{lse}(\beta,\mathbf{z}) &= \frac{1}{\beta}\log\left(\sum_{i=1}^{K}\exp(\beta\,\mathbf{z}_{i})\right) \\
\end{align*}
$$
and $\beta\in\mathbb{R}_{>0}$ is an inverse temperature parameter that controls the sharpness of the distribution. Finally, $M$ is the largest Euclidean memory norm, i.e., $M = \max_{i=1,\dots,K}\lVert\mathbf{m}_{i}\rVert_2$. The constants $\frac{1}{\beta}\log(K)$ and $\frac{1}{2}M^2$ ensure the energy remains bounded and comparable across different configurations.

The __vector__ $\mathbf{X}^{\top}\mathbf{s}$ computes the similarity (dot product) between the current state $\mathbf{s}$ and each stored memory, producing a $K$-dimensional vector of similarities. The log-sum-exp function then aggregates these similarities in a smooth, differentiable manner.

<div>
    <center>
        <img src="figs/Fig-Matrix-Vector-Right-Ab-product-NeedToRedrawThis.png" width="580"/>
    </center>
</div>

## Algorithm: Memory retrieval
The user provides a set of memory vectors $\mathbf{X} = \left\{\mathbf{m}_{1}, \mathbf{m}_{2}, \ldots, \mathbf{m}_{K}\right\}$, where $\mathbf{m}_{i} \in \mathbb{R}^{N}$ is a memory vector of size $N$ and $K$ is the number of memory vectors. Further, the user provides an initial _partial memory_ $\mathbf{s}_{\circ} \in \mathbb{R}^{N}$, which is a vector of size $N$, and specifies the _inverse temperature_ $\beta$ of the system. Define the retrieval map:
$$
\mathbf{T}(\mathbf{s}) := \mathbf{X}\,\operatorname{softmax}(\beta\mathbf{X}^{\top}\mathbf{s}).
$$
where $\mathbf{s}^{t+1} = \mathbf{T}(\mathbf{s}^{t})$ is the update rule for retrieving a memory from the network. For this energy model, the gradient is given by:
$$
\nabla E(\mathbf{s}) = \mathbf{s} - \mathbf{T}(\mathbf{s}).
$$
So the default retrieval update (using gradient descent) is given by:
$$
\mathbf{s}^{t+1}=(1-\eta)\mathbf{s}^{t}+\eta\mathbf{T}(\mathbf{s}^{t})=\mathbf{s}^{t}-\eta\nabla E(\mathbf{s}^{t}),\quad 0<\eta\leq 1.
$$
In this lecture and theorem statement below, we use the default map with $\eta=1$.

__Initialize__ the network with the memory matrix $\mathbf{X}$ and inverse temperature $\beta\in\mathbb{R}_{>0}$. Set the current state $\mathbf{s} \gets \mathbf{s}_{\circ}$, initialize the iteration counter $t \gets 1$, maximum iterations $\texttt{maxiter}$, and set convergence flag $\texttt{converged} \gets \texttt{false}$, tolerance $\epsilon > 0$, and previous-probability placeholder $\mathbf{p}_{\text{prev}}\gets\texttt{nothing}$.

> **Parameter Guidelines**: Common choices are `maxiter = 1000` and $\epsilon$ = `1e-6`. Modern Hopfield networks typically converge within 10–100 iterations, making `maxiter = 1000` a conservative upper bound. The tolerance $\epsilon$ = `1e-6` provides good precision for most applications while avoiding numerical precision issues.

While not $\texttt{converged}$ and $t \leq \texttt{maxiter}$ __do__:
   1. Compute the _current_ similarity vector $\mathbf{z} = \mathbf{X}^{\top}\mathbf{s}$, where each element $z_i = \mathbf{m}_i^{\top}\mathbf{s}$ represents the similarity between the current state and memory $i$.
   2. Compute the _current_ probability vector $\mathbf{p} = \texttt{softmax}(\beta\cdot\mathbf{z})$ where $\texttt{softmax}(\mathbf{u})_i = \frac{\exp(u_i)}{\sum_{j=1}^{K}\exp(u_j)}$.
   3. Compute the _next_ state vector $\mathbf{s}^{\prime} = \mathbf{X}\mathbf{p}=\mathbf{T}(\mathbf{s})$ using the current probability vector $\mathbf{p}$ and the memory matrix $\mathbf{X}$. This step computes a weighted sum of the memory vectors based on the probabilities.
   4. **Check for convergence**: If $\lVert \mathbf{s}^{\prime} - \mathbf{s}\rVert_{2} \leq \epsilon$, then set $\texttt{converged} \gets \texttt{true}$.
      - **Alternative (from the second iteration onward)**: If $\mathbf{p}_{\text{prev}}\neq\texttt{nothing}$ and $\lVert \mathbf{p} - \mathbf{p}_{\text{prev}}\rVert_{1} \leq \epsilon_p$, where $\epsilon_p$ is the convergence tolerance for probabilities (default: $\epsilon_{p}$ = `1e-8`) and $\lVert\star\rVert_{1}$ is the L1-norm, then set $\texttt{converged} \gets \texttt{true}$.
   5. **Update state**: Set $\mathbf{p}_{\text{prev}}\gets\mathbf{p}$, then update $\mathbf{s} \gets\mathbf{s}^{\prime}$ and increment $t \gets t + 1$.

> **Note**: The softmax function in step 2 is directly related to the log-sum-exp function in the energy formulation. Specifically, the gradient of the LSE with respect to $\mathbf{s}$ yields the softmax-weighted combination of memories used in the update rule.

Let's look at an example to illustrate these ideas.

> __Example:__
>
> [▶ Analyze a modern Hopfield Network](CHEME-5800-L15c-Example-ModernHopfieldNetwork-Fall-2025.ipynb). In this example, we analyze an example of a modern Hopfield Network to understand how it encodes and retrieves gray scale (continuous) patterns. This example builds on the concepts introduced in the previous lecture and demonstrates the application of modern Hopfield networks in continuous associative memory tasks.

___

## Convergence
Modern Hopfield networks have strong convergence structure, but it helps to separate two questions: what is guaranteed theoretically, and what is commonly observed in practice. Let's start with the theoretical guarantee.

Before the formal statement, here is the intuition: each retrieval step decreases (or leaves unchanged) the energy and moves the state toward a self-consistent memory mixture. So trajectories cannot wander indefinitely; they settle toward fixed-point behavior, though not necessarily in a fixed number of steps.

Equivalent viewpoint: retrieval is gradient descent on the same energy. Since $\nabla E(\mathbf{s})=\mathbf{s}-\mathbf{T}(\mathbf{s})$, the default update
$$
\mathbf{s}^{t+1}=\mathbf{T}(\mathbf{s}^{t})=\mathbf{s}^{t}-\nabla E(\mathbf{s}^{t})
$$
is gradient descent with step size $1$. The relaxed family
$$
\mathbf{s}^{t+1}=(1-\eta)\mathbf{s}^{t}+\eta\mathbf{T}(\mathbf{s}^{t})=\mathbf{s}^{t}-\eta\nabla E(\mathbf{s}^{t}),\quad 0<\eta\leq 1
$$
is also useful in practice. The theorem below is stated for the default map $\eta=1$.

> __Convergence Theorem (Modern Hopfield, continuous-state LSE energy):__
>
> Let $N,K\in\mathbb{Z}_{\geq 1}$, where $N$ is the state dimension (number of nodes/features) and $K$ is the number of stored memories. Let $\beta>0$ denote the inverse temperature parameter. Let
> $\mathbf{X}=[\mathbf{m}_1,\ldots,\mathbf{m}_K]\in\mathbb{R}^{N\times K}$ be the memory matrix, where column $\mathbf{m}_i\in\mathbb{R}^{N}$ is stored memory $i$. Define
> $M:=\max_{i=1,\ldots,K}\lVert\mathbf{m}_i\rVert_2$, i.e., $M$ is the largest memory norm. For any similarity vector $\mathbf{z}\in\mathbb{R}^{K}$ and any state $\mathbf{s}\in\mathbb{R}^{N}$, define
> $$
> \begin{aligned}
> \operatorname{lse}_{\beta}(\mathbf{z})
> &:= \frac{1}{\beta}\log\left(\sum_{i=1}^{K} e^{\beta z_i}\right),\\
> E(\mathbf{s})
> &:= -\operatorname{lse}_{\beta}(\mathbf{X}^{\top}\mathbf{s}) + \frac{1}{2}\lVert\mathbf{s}\rVert_2^2 + \frac{1}{\beta}\log K + \frac{1}{2}M^2,\\
> \mathbf{T}(\mathbf{s})
> &:= \mathbf{X}\,\operatorname{softmax}(\beta\mathbf{X}^{\top}\mathbf{s}).
> \end{aligned}
> $$
> Here $\operatorname{softmax}(\mathbf{u})_i := \frac{e^{u_i}}{\sum_{j=1}^{K}e^{u_j}}$ for $\mathbf{u}\in\mathbb{R}^{K}$ and $i=1,\ldots,K$. Also, $\operatorname{lse}_{\beta}$ is the smooth maximum over similarities, $E$ is the Lyapunov energy, and $\mathbf{T}$ is the retrieval update map.
> If $\mathbf{s}^{t+1}=\mathbf{T}(\mathbf{s}^{t})$, then for every $t$,
> $$
> E(\mathbf{s}^{t+1}) \le E(\mathbf{s}^{t}) - \frac{1}{2}\lVert\mathbf{s}^{t+1}-\mathbf{s}^{t}\rVert_2^2,
> $$
> and for every $\mathbf{s}\in\mathbb{R}^{N}$,
> $$
> E(\mathbf{s}) \ge \frac{1}{2}\left(\lVert\mathbf{s}\rVert_2 - M\right)^2 \ge 0.
> $$
> Therefore $E(\mathbf{s}^{t})$ never increases and converges, $\lVert\mathbf{s}^{t+1}-\mathbf{s}^{t}\rVert_2\to 0$ as $t\to\infty$, and every limit point $\mathbf{s}^{\ast}$ is a fixed point,
> $$
> \mathbf{T}(\mathbf{s}^{\ast})=\mathbf{s}^{\ast},
> $$
> equivalently $\nabla E(\mathbf{s}^{\ast})=\mathbf{0}$. This theorem gives asymptotic convergence structure; it does not claim finite-step convergence for every initialization. For the full derivation, see [Advanced: Why Modern Hopfield Networks Converge (Continuous LSE Model)](CHEME-5820-L6c-Advanced-Modern-HopfieldConvergence-Attention-Proof-Spring-2026.ipynb).

The theorem gives the guaranteed behavior. In practice, many retrieval trajectories are much faster than the asymptotic guarantee suggests, but speed depends on model and data geometry.

> **Convergence in Practice**
>
> In many practical runs, modern Hopfield retrieval converges quickly (often within a small number of iterations). However, the iteration count can increase when the initial state is far from all memories or when stored memories are strongly correlated.
> The frequently cited 1-5 iteration behavior is an empirical observation in favorable settings, not part of the formal theorem above.
>
> **Factors affecting practical speed:**
> - **Inverse temperature $\beta$**: Higher $\beta$ (lower temperature) makes softmax attention more selective, so trajectories usually commit faster to a dominant memory and settle in fewer steps; if $\beta$ is too large, basins become narrow and noisy starts are less forgiving.
> - **Memory separation**: When stored memories are well separated (and similarly scaled), softmax tends to produce a clear winner early, giving larger early energy drops and faster convergence; highly correlated memories split probability mass and delay commitment.
> - **Initialization quality**: States that start near a target basin move quickly into fixed-point settling, while distant or heavily corrupted starts spend more iterations in diffuse matching and may converge to a competing fixed point.
>
> **If convergence is slow in practice:** Try reducing $\beta$, normalizing memory vectors, increasing `maxiter`, or restarting from a different initialization.

___

## Lab
In the lab, we will implement the modern Hopfield network retrieval algorithm and analyze its behavior on a simple financial associative-memory task. We'll sweep the inverse temperature parameter $\beta$ to move between sharp recall of a single memory (high $\beta$, low temperature) and soft generation of a mixture of memories (low $\beta$, high temperature).

## Summary

In this lecture, we explored modern Hopfield networks as a powerful generalization of classical associative memory systems:

> __Key takeaways:__
>
> * **Modern Hopfield Energy Function**: Modern Hopfield networks use the log-sum-exp energy function to store continuous patterns rather than binary states. The inverse temperature parameter $\beta$ controls the sharpness of memory retrieval, with higher values producing sharper, more decisive pattern selection through the softmax operation.
> * **Exponential Storage and Convergence**: Modern Hopfield networks achieve exponentially larger storage capacity than classical networks and exhibit exponential convergence (typically 1–5 iterations) through softmax-weighted updates. This dramatic improvement over the classical $0.138N$ storage limit and polynomial convergence makes them practical for high-dimensional continuous data.
> * **Connection to Attention Mechanisms**: The modern Hopfield update rule is mathematically equivalent to single-head attention in transformer architectures. This connection reveals that attention mechanisms can be understood as associative memory retrieval, bridging classical neural network theory with contemporary deep learning.


Modern Hopfield networks provide both theoretical insight into attention mechanisms and practical tools for continuous associative memory tasks in high-dimensional spaces.
___

## References
Background reading for this lecture (and the associated lab) can be found from the following sources:
* [Krotov, D., & Hopfield, J.J. (2016). Dense Associative Memory for Pattern Recognition. ArXiv, abs/1606.01164.](https://arxiv.org/abs/1606.01164)
* [Demircigil, M., Heusel, J., Löwe, M., Upgang, S., & Vermet, F. (2017). On a Model of Associative Memory with Huge Storage Capacity. Journal of Statistical Physics, 168, 288 - 299.](https://arxiv.org/abs/1702.01929)
* [Ramsauer, H., Schafl, B., Lehner, J., Seidl, P., Widrich, M., Gruber, L., Holzleitner, M., Pavlovi'c, M., Sandve, G.K., Greiff, V., Kreil, D.P., Kopp, M., Klambauer, G., Brandstetter, J., & Hochreiter, S. (2020). Hopfield Networks is All You Need. ArXiv, abs/2008.02217.](https://arxiv.org/abs/2008.02217)
* [Krotov, D., & Hopfield, J.J. (2020). Large Associative Memory Problem in Neurobiology and Machine Learning. ArXiv, abs/2008.06996.](https://arxiv.org/abs/2008.06996)

The following blog post is also helpful: [Hopfield Networks is All You Need Blog, GitHub.io](https://ml-jku.github.io/hopfield-layers/)
___